# Faster R-CNN notes

This is the notebook I used for the Faster R-CNN baseline on the same dataset.

Flow in this file:
- copy the data from Drive
- make the train/validation split
- build the dataset and detector
- train with mixed precision
- save checkpoints and run extra analysis

## 1) Setup
Mount Drive first, then install/import the packages for this run.

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# install a few packages
!pip -q install -U "torch>=2.0" "torchvision>=0.15" pycocotools
!pip -q install seaborn matplotlib pandas scikit-learn tqdm

In [ ]:
import os

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# main paths + settings
import os, shutil, json, random
from glob import glob

DRIVE_TRAIN_ZIP = '/content/drive/Shareddrives/Workspace/NKla/dataset.zip'
DRIVE_TEST_ZIP  = '/content/drive/Shareddrives/Workspace/NKla/test.zip'

EXP_LOG_DIR = '/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments_frcnn'
LOCAL_ROOT  = '/content/temp_dataset'

TARGET_SIZE = 11000

CLASS_NAMES = [
    'Ascaris lumbricoides', 'Capillaria philippinensis', 'Enterobius vermicularis',
    'Fasciolopsis buski', 'Hookworm egg', 'Hymenolepis diminuta',
    'Hymenolepis nana', 'Opisthorchis viverrine', 'Paragonimus spp',
    'Taenia spp. egg', 'Trichuris trichiura'
]

os.makedirs(EXP_LOG_DIR, exist_ok=True)
print("EXP_LOG_DIR =", EXP_LOG_DIR)

In [ ]:
# copy the zip files to local storage
def prepare_local_data():
    # I copy everything to local storage first because loading is usually smoother there.
    local_train_dir = os.path.join(LOCAL_ROOT, 'train')
    if os.path.exists(local_train_dir):
        shutil.rmtree(local_train_dir)
    os.makedirs(local_train_dir, exist_ok=True)

    print("copying train zip from Drive...")
    shutil.copy(DRIVE_TRAIN_ZIP, '/content/train_temp.zip')
    print("extracting train data...")
    shutil.unpack_archive('/content/train_temp.zip', local_train_dir)
    os.remove('/content/train_temp.zip')

    if os.path.exists(os.path.join(local_train_dir, 'data')):
        print("Renaming 'data' to 'images'...")
        os.rename(os.path.join(local_train_dir, 'data'), os.path.join(local_train_dir, 'images'))

    local_test_dir = os.path.join(LOCAL_ROOT, 'test')
    if os.path.exists(local_test_dir):
        shutil.rmtree(local_test_dir)
    os.makedirs(local_test_dir, exist_ok=True)

    print("copying test zip from Drive...")
    shutil.copy(DRIVE_TEST_ZIP, '/content/test_temp.zip')
    print("extracting test data...")
    shutil.unpack_archive('/content/test_temp.zip', local_test_dir)
    os.remove('/content/test_temp.zip')

    if os.path.exists(os.path.join(local_test_dir, 'data')):
        print("Renaming 'data' to 'images'...")
        os.rename(os.path.join(local_test_dir, 'data'), os.path.join(local_test_dir, 'images'))

    print("done Data layout fixed:", LOCAL_ROOT)
    print("Train images:", len(os.listdir(os.path.join(local_train_dir, 'images'))))
    print("Test  images:", len(os.listdir(os.path.join(local_test_dir, 'images'))))
    return local_train_dir, local_test_dir

LOCAL_TRAIN_DIR, LOCAL_TEST_DIR = prepare_local_data()

## 2) Data split and loading
This section finds the COCO files, keeps the target data size, and builds the dataloaders.

In [ ]:
# find the annotation files and make train/val split
import os, json
from glob import glob
from sklearn.model_selection import train_test_split

def find_coco_json(root_dir):
    # check common locations first
    cands = []
    cands += glob(os.path.join(root_dir, "annotations", "*.json"))
    cands += glob(os.path.join(root_dir, "**", "*.json"), recursive=True)
    # keep only files that look like COCO annotations
    for p in cands:
        try:
            with open(p, "r", encoding="utf-8") as f:
                d = json.load(f)
            if isinstance(d, dict) and "images" in d and "annotations" in d and "categories" in d:
                return p
        except:
            pass
    raise FileNotFoundError(f"Could not find a COCO json under: {root_dir}")

TRAIN_IMG_DIR = os.path.join(LOCAL_TRAIN_DIR, "images")
TEST_IMG_DIR  = os.path.join(LOCAL_TEST_DIR, "images")

TRAIN_JSON_PATH = find_coco_json(LOCAL_TRAIN_DIR)
TEST_JSON_PATH  = find_coco_json(LOCAL_TEST_DIR)

print("TRAIN_JSON_PATH:", TRAIN_JSON_PATH)
print("TEST_JSON_PATH :", TEST_JSON_PATH)
print("TRAIN_IMG_DIR  :", TRAIN_IMG_DIR)
print("TEST_IMG_DIR   :", TEST_IMG_DIR)

with open(TRAIN_JSON_PATH, "r", encoding="utf-8") as f:
    raw_coco = json.load(f)

images = raw_coco["images"]
annotations = raw_coco["annotations"]
categories = raw_coco["categories"]

# keep one fixed subset size for this run
img_ids = [img["id"] for img in images][:TARGET_SIZE]
train_ids, val_ids = train_test_split(img_ids, test_size=0.2, random_state=42)

train_ids = set(train_ids)
val_ids   = set(val_ids)

images_train = [img for img in images if img["id"] in train_ids]
images_val   = [img for img in images if img["id"] in val_ids]

anns_train = [a for a in annotations if a["image_id"] in train_ids]
anns_val   = [a for a in annotations if a["image_id"] in val_ids]

COCO_TRAIN_JSON = os.path.join(EXP_LOG_DIR, f"coco_train_{TARGET_SIZE}.json")
COCO_VAL_JSON   = os.path.join(EXP_LOG_DIR, f"coco_val_{TARGET_SIZE}.json")

with open(COCO_TRAIN_JSON, "w", encoding="utf-8") as f:
    json.dump({"images": images_train, "annotations": anns_train, "categories": categories}, f)
with open(COCO_VAL_JSON, "w", encoding="utf-8") as f:
    json.dump({"images": images_val, "annotations": anns_val, "categories": categories}, f)

print("saved:", COCO_TRAIN_JSON)
print("saved:", COCO_VAL_JSON)
print("Train imgs:", len(images_train), "Val imgs:", len(images_val))

In [ ]:
# dataset class + dataloaders
import torch
from torch.utils.data import DataLoader
from PIL import Image
from torchvision.transforms import functional as F
from pycocotools.coco import COCO

def collate_batch(batch):
    return tuple(zip(*batch))

class EggCocoDataset(torch.utils.data.Dataset):
    def __init__(self, images_dir, ann_json_path, require_anns=False):
        self.images_dir = images_dir
        self.coco = COCO(ann_json_path)

        all_ids = list(sorted(self.coco.imgs.keys()))
        if require_anns:
            self.img_ids = [i for i in all_ids if len(self.coco.getAnnIds(imgIds=[i])) > 0]
        else:
            self.img_ids = all_ids

        cat_ids = list(sorted(self.coco.getCatIds()))
        self.catid_to_contig = {cid: i+1 for i, cid in enumerate(cat_ids)}
        self.contig_to_catid = {v: k for k, v in self.catid_to_contig.items()}

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]

        img_path = os.path.join(self.images_dir, info["file_name"])
        if not os.path.exists(img_path):
            # fallback in case the extension is different
            base = os.path.splitext(img_path)[0]
            for ext in [".jpg", ".jpeg", ".png", ".bmp"]:
                if os.path.exists(base + ext):
                    img_path = base + ext
                    break

        img = Image.open(img_path).convert("RGB")
        img = F.to_tensor(img)

        ann_ids = self.coco.getAnnIds(imgIds=[img_id])
        anns = self.coco.loadAnns(ann_ids)

        boxes, labels, areas, iscrowd = [], [], [], []
        for a in anns:
            x, y, w, h = a["bbox"]
            if w <= 1 or h <= 1:
                continue
            boxes.append([x, y, x+w, y+h])
            labels.append(self.catid_to_contig.get(a["category_id"], 0))
            areas.append(a.get("area", w*h))
            iscrowd.append(a.get("iscrowd", 0))

        if len(boxes) == 0:
            boxes = torch.zeros((0,4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            areas = torch.zeros((0,), dtype=torch.float32)
            iscrowd = torch.zeros((0,), dtype=torch.int64)
        else:
            boxes = torch.tensor(boxes, dtype=torch.float32)
            labels = torch.tensor(labels, dtype=torch.int64)
            areas = torch.tensor(areas, dtype=torch.float32)
            iscrowd = torch.tensor(iscrowd, dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([img_id], dtype=torch.int64),
            "area": areas,
            "iscrowd": iscrowd
        }
        return img, target

train_ds = EggCocoDataset(TRAIN_IMG_DIR, COCO_TRAIN_JSON, require_anns=True)
val_ds   = EggCocoDataset(TRAIN_IMG_DIR, COCO_VAL_JSON,   require_anns=True)
test_ds  = EggCocoDataset(TEST_IMG_DIR,  TEST_JSON_PATH,  require_anns=False)

BATCH_SIZE = 1        # <-- OOM FIX
BATCH_EVAL = 1
NUM_WORKERS = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, collate_batch=collate_batch,
                          pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
val_loader   = DataLoader(val_ds, batch_size=BATCH_EVAL, shuffle=False,
                          num_workers=NUM_WORKERS, collate_batch=collate_batch,
                          pin_memory=True, persistent_workers=(NUM_WORKERS > 0))
test_loader  = DataLoader(test_ds, batch_size=BATCH_EVAL, shuffle=False,
                          num_workers=NUM_WORKERS, collate_batch=collate_batch,
                          pin_memory=True, persistent_workers=(NUM_WORKERS > 0))

print("sizes:", len(train_ds), len(val_ds), len(test_ds))

## 3) Model and training
Here I build the Faster R-CNN model, train it, and save the history/checkpoint files.

In [ ]:
# build the Faster R-CNN model
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

use_pretrained = True

MIN_SIZE = 600
MAX_SIZE = 1000

if use_pretrained:
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights="DEFAULT", min_size=MIN_SIZE, max_size=MAX_SIZE
    )
else:
    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone=None, min_size=MIN_SIZE, max_size=MAX_SIZE
    )

# swap the classifier head for my class count
detector_num_classes = len(train_ds.catid_to_contig) + 1
in_features = model.roi_heads.box_predictor.cls_score.in_features
model.roi_heads.box_predictor = FastRCNNPredictor(in_features, detector_num_classes)

# trim a few proposal settings to make memory usage easier
# ROI sampling
model.roi_heads.batch_size_per_image = 128  # default 512

# RPN proposal settings across torchvision versions
rpn = model.rpn

if hasattr(rpn, "_pre_nms_top_n") and isinstance(rpn._pre_nms_top_n, dict):
    rpn._pre_nms_top_n["training"] = 1000
    rpn._pre_nms_top_n["testing"]  = 1000
elif hasattr(rpn, "pre_nms_top_n") and isinstance(rpn.pre_nms_top_n, dict):
    rpn.pre_nms_top_n["training"] = 1000
    rpn.pre_nms_top_n["testing"]  = 1000
else:
    print("Cannot set pre_nms_top_n in this torchvision version (will skip).")

if hasattr(rpn, "_post_nms_top_n") and isinstance(rpn._post_nms_top_n, dict):
    rpn._post_nms_top_n["training"] = 500
    rpn._post_nms_top_n["testing"]  = 500
elif hasattr(rpn, "post_nms_top_n") and isinstance(rpn.post_nms_top_n, dict):
    rpn.post_nms_top_n["training"] = 500
    rpn.post_nms_top_n["testing"]  = 500
else:
    print("Cannot set post_nms_top_n in this torchvision version (will skip).")

model.to(device)

# optional check
print("roi_heads.batch_size_per_image =", model.roi_heads.batch_size_per_image)
if hasattr(rpn, "_pre_nms_top_n"):
    print("rpn._pre_nms_top_n =", rpn._pre_nms_top_n)
if hasattr(rpn, "_post_nms_top_n"):
    print("rpn._post_nms_top_n =", rpn._post_nms_top_n)

In [ ]:
# train + validation helpers
import torch, numpy as np
from tqdm.auto import tqdm

ACCUM_STEPS = 4

def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    losses = []
    pbar = tqdm(data_loader, desc=f"Epoch {epoch} [train]")

    optimizer.zero_grad(set_to_none=True)

    for step, (images, targets) in enumerate(pbar, 1):
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()} for t in targets]

        with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=(device.type == "cuda")):
            loss_dict = model(images, targets)
            loss = sum(l for l in loss_dict.values()) / ACCUM_STEPS

        loss.backward()

        if step % ACCUM_STEPS == 0:
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        losses.append(loss.item() * ACCUM_STEPS)
        pbar.set_postfix(loss=float(np.mean(losses)))

        del images, targets, loss_dict, loss

    if (step % ACCUM_STEPS) != 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    if device.type == "cuda":
        torch.cuda.empty_cache()
    return float(np.mean(losses))

@torch.no_grad()
def coco_evaluate(model, data_loader, coco_gt, contig_to_catid, device):
    model.eval()
    coco_results = []

    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = [img.to(device, non_blocking=True) for img in images]
        outputs = model(images)

        for out, tgt in zip(outputs, targets):
            image_id = int(tgt["image_id"].item())
            boxes  = out["boxes"].detach().cpu().numpy()
            scores = out["scores"].detach().cpu().numpy()
            labels = out["labels"].detach().cpu().numpy()

            for box, score, lab in zip(boxes, scores, labels):
                x1, y1, x2, y2 = box.tolist()
                w, h = x2 - x1, y2 - y1
                if w <= 0 or h <= 0:
                    continue
                coco_results.append({
                    "image_id": image_id,
                    "category_id": int(contig_to_catid.get(int(lab), int(lab))),
                    "bbox": [x1, y1, w, h],
                    "score": float(score),
                })

        del outputs, images, targets
        if device.type == "cuda":
            torch.cuda.empty_cache()

    if len(coco_results) == 0:
        return {"AP": 0.0, "AP50": 0.0, "AR100": 0.0}

    coco_dt = coco_gt.loadRes(coco_results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()

    stats = coco_eval.stats
    return {"AP": float(stats[0]), "AP50": float(stats[1]), "AR100": float(stats[8])}

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.0025, momentum=0.9, weight_decay=0.0005)

EPOCHS = 10
coco_val_gt  = val_ds.coco
coco_test_gt = test_ds.coco

def fix_coco_gt(coco):
    changed = False
    for a in coco.dataset.get("annotations", []):
        if "iscrowd" not in a:
            a["iscrowd"] = 0
            changed = True
        if "area" not in a:
            bb = a.get("bbox", None)
            if bb and len(bb) == 4:
                a["area"] = float(bb[2] * bb[3])
                changed = True
    if changed:
        coco.createIndex()
    return coco

coco_val_gt  = fix_coco_gt(coco_val_gt)
coco_test_gt = fix_coco_gt(coco_test_gt)
print("Patched coco_gt: added iscrowd/area if missing")

history = []
for epoch in range(1, EPOCHS+1):
    train_loss = train_one_epoch(model, optimizer, train_loader, device, epoch)
    if device.type == "cuda":
        torch.cuda.empty_cache()
    val_metrics = coco_evaluate(model, val_loader, coco_val_gt, train_ds.contig_to_catid, device)

    row = {"epoch": epoch, "train_loss": train_loss, **{f"val_{k}": v for k, v in val_metrics.items()}}
    history.append(row)
    print("Epoch", epoch, row)

test_metrics = coco_evaluate(model, test_loader, coco_test_gt, train_ds.contig_to_catid, device)
print("TEST metrics:", test_metrics)

In [ ]:
# save history + checkpoint
import os, torch
import pandas as pd

df = pd.DataFrame(history)
hist_path = os.path.join(EXP_LOG_DIR, f"frcnn_history_{TARGET_SIZE}.csv")
df.to_csv(hist_path, index=False)
print("history saved to:", hist_path)

metric_key = "val_AP"
best_row = df.loc[df[metric_key].idxmax()]
best_epoch = int(best_row["epoch"])
best_score = float(best_row[metric_key])
last_epoch = int(df["epoch"].iloc[-1])

print(f"Best by {metric_key}: epoch={best_epoch}, {metric_key}={best_score:.4f}")
print(f"Last epoch: {last_epoch}")

# save the current in-memory model
BEST_PATH = os.path.join(EXP_LOG_DIR, f"best_frcnn{TARGET_SIZE}.pt")
torch.save({
    "epoch": last_epoch,
    "metric_best_epoch": best_epoch,
    "metric_best_value": best_score,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "num_classes": detector_num_classes,
    "detector_num_classes": detector_num_classes,
    "MIN_SIZE": MIN_SIZE,
    "MAX_SIZE": MAX_SIZE,
    "catid_to_contig": train_ds.catid_to_contig,
    "contig_to_catid": train_ds.contig_to_catid,
    "CLASS_NAMES": CLASS_NAMES,
}, BEST_PATH)

print("checkpoint saved to:", BEST_PATH)

if best_epoch != last_epoch:
    print("best metric was not from the last epoch")

## 4) Extra analysis
These last cells are the extra checks I used for confusion matrices, timing, and comparison with the YOLO runs.

In [ ]:
# Faster R-CNN confusion matrix
import os
import numpy as np
import torch
from tqdm.auto import tqdm
from torchvision.ops import box_iou
import matplotlib.pyplot as plt
import pandas as pd

def detection_confusion_matrix(
    model, data_loader, detector_num_classes, device,
    iou_thr=0.5, conf_thr=0.25, max_images=None
):
    """
    detector_num_classes = K+1 (including background=0 in model head)
    returns (K+1)x(K+1) matrix with last index = background
    rows = GT, cols = Pred
    """
    K = detector_num_classes - 1
    BG = K
    cm = np.zeros((K+1, K+1), dtype=np.int64)

    model.eval()
    seen = 0

    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc="ConfusionMatrix"):
            images = [img.to(device, non_blocking=True) for img in images]
            outputs = model(images)

            for out, tgt in zip(outputs, targets):
                gt_boxes  = tgt["boxes"].cpu()
                gt_labels = tgt["labels"].cpu()

                pb = out["boxes"].detach().cpu()
                ps = out["scores"].detach().cpu()
                pl = out["labels"].detach().cpu()

                keep = ps >= conf_thr
                pb, ps, pl = pb[keep], ps[keep], pl[keep]

                if ps.numel() > 0:
                    order = torch.argsort(ps, descending=True)
                    pb, pl = pb[order], pl[order]

                matched_gt = torch.zeros((gt_boxes.size(0),), dtype=torch.bool)

                ious = None
                if pb.numel() > 0 and gt_boxes.numel() > 0:
                    ious = box_iou(pb, gt_boxes)

                for p in range(pb.size(0)):
                    pred_cls = int(pl[p].item()) - 1

                    if gt_boxes.size(0) == 0:
                        cm[BG, pred_cls] += 1
                        continue

                    iou_row = ious[p].clone()
                    iou_row[matched_gt] = -1
                    best_iou, best_g = torch.max(iou_row, dim=0)

                    if best_iou >= iou_thr:
                        matched_gt[best_g] = True
                        gt_cls = int(gt_labels[best_g].item()) - 1
                        cm[gt_cls, pred_cls] += 1
                    else:

                        cm[BG, pred_cls] += 1

                for g in range(gt_boxes.size(0)):
                    if not matched_gt[g]:
                        gt_cls = int(gt_labels[g].item()) - 1
                        cm[gt_cls, BG] += 1

            seen += len(outputs)
            if max_images is not None and seen >= max_images:
                break

    return cm

IOU_THR  = 0.75
CONF_THR = 0.75
MAX_IMAGES = None

cm = detection_confusion_matrix(
    model, val_loader, detector_num_classes, device,
    iou_thr=IOU_THR, conf_thr=CONF_THR, max_images=MAX_IMAGES
)

labels = list(CLASS_NAMES) + ["__background__"]

cm_csv = os.path.join(EXP_LOG_DIR, f"frcnn_confmat_val_{TARGET_SIZE}_iou{IOU_THR}_conf{CONF_THR}.csv")
pd.DataFrame(cm, index=labels, columns=labels).to_csv(cm_csv)
print("saved:", cm_csv)

plt.figure(figsize=(10,8))
plt.imshow(cm, interpolation="nearest")
plt.title(f"Detection Confusion Matrix (IoU={IOU_THR}, conf>={CONF_THR})")
plt.xticks(range(len(labels)), labels, rotation=90)
plt.yticks(range(len(labels)), labels)
plt.colorbar()
plt.tight_layout()
plt.show()

cm_norm = cm.astype(np.float64)
row_sum = cm_norm.sum(axis=1, keepdims=True)
cm_norm = np.divide(cm_norm, row_sum, out=np.zeros_like(cm_norm), where=row_sum!=0)

plt.figure(figsize=(10,8))
plt.imshow(cm_norm, interpolation="nearest")
plt.title(f"Normalized (by GT row) (IoU={IOU_THR}, conf>={CONF_THR})")
plt.xticks(range(len(labels)), labels, rotation=90)
plt.yticks(range(len(labels)), labels)
plt.colorbar()
plt.tight_layout()
plt.show()

K = len(CLASS_NAMES)
TP = np.diag(cm)[:K]
FP = cm[:, :K].sum(axis=0) - TP
FN = cm[:K, :].sum(axis=1) - TP

precision = TP / np.maximum(TP + FP, 1)
recall    = TP / np.maximum(TP + FN, 1)

metrics_df = pd.DataFrame({
    "class": CLASS_NAMES,
    "TP": TP, "FP": FP, "FN": FN,
    "precision": precision,
    "recall": recall
})
metrics_path = os.path.join(EXP_LOG_DIR, f"frcnn_pr_val_{TARGET_SIZE}_iou{IOU_THR}_conf{CONF_THR}.csv")
metrics_df.to_csv(metrics_path, index=False)
print("saved:", metrics_path)
metrics_df

In [ ]:
!pip -q install -U ultralytics

In [ ]:
import os, gc, torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from torchvision.ops import box_iou
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from ultralytics import YOLO

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

YOLO8_BEST  = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments/runs/yolov8s_11000/weights/best.pt"
YOLO11_BEST = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments/runs/yolo11s_11000/weights/best.pt"
FRCNN_BEST  = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments_frcnn/best_frcnn11000.pt"

EVAL_DS  = test_ds
IMG_ROOT = TEST_IMG_DIR

CONF_THR = 0.25
IOU_THR  = 0.5
MAX_IMAGES = None

CLASS_NAMES = CLASS_NAMES
K = len(CLASS_NAMES)
name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}

def map_yolo_cls_to_classidx(yolo_model):
    mp = {}
    for yid, nm in yolo_model.names.items():
        mp[int(yid)] = name_to_idx.get(nm, None)
    return mp

def frcnn_label_to_classidx(label_contig, ds):
    lab = int(label_contig)
    if hasattr(ds, "contig_to_catid") and lab in ds.contig_to_catid:
        catid = ds.contig_to_catid[lab]
        nm = ds.coco.cats.get(catid, {}).get("name", None)
        if nm in name_to_idx:
            return name_to_idx[nm]
    x = lab - 1
    return x if 0 <= x < K else None

def load_frcnn_ckpt(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt.get("model_state_dict", ckpt.get("model_state", None))
    if state is None:
        raise KeyError("FRCNN ckpt missing model_state_dict/model_state")

    detector_num_classes = int(ckpt.get("detector_num_classes", K + 1))
    MIN_SIZE = int(ckpt.get("MIN_SIZE", 600))
    MAX_SIZE = int(ckpt.get("MAX_SIZE", 1000))

    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone=None, min_size=MIN_SIZE, max_size=MAX_SIZE
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, detector_num_classes)
    model.load_state_dict(state)
    model.to(device).eval()
    return model, detector_num_classes

def compute_image_ious_per_class(gt_boxes, gt_cls, pb, pl, ps, conf_thr, iou_thr):

    keep = [i for i,s in enumerate(ps) if (s >= conf_thr and pl[i] is not None)]
    pb = pb[keep] if len(keep) else np.zeros((0,4), dtype=np.float32)
    pl = [pl[i] for i in keep]
    ps = [ps[i] for i in keep]

    per_class_vals = [[] for _ in range(K)]

    for c in range(K):
        gt_idx = np.where(np.array(gt_cls) == c)[0]
        if gt_idx.size == 0:
            continue

        vals = [0.0]*int(gt_idx.size)

        pr_idx = [j for j,pc in enumerate(pl) if pc == c]
        if len(pr_idx) > 0:
            gtb = torch.tensor(gt_boxes[gt_idx], dtype=torch.float32)
            prb = torch.tensor(pb[pr_idx], dtype=torch.float32)
            iou_mat = box_iou(prb, gtb)  # (P,G)

            used_p=set(); used_g=set()
            flat=iou_mat.flatten()
            order=torch.argsort(flat, descending=True)
            P,G = iou_mat.shape

            for idx_flat in order.tolist():
                p = idx_flat // G
                g = idx_flat %  G
                if p in used_p or g in used_g:
                    continue
                v = float(iou_mat[p,g].item())
                if v < iou_thr:
                    break
                used_p.add(p); used_g.add(g)
                vals[g] = v

        per_class_vals[c].extend(vals)

    return per_class_vals

@torch.no_grad()
def eval_miou_yolo(yolo_model, ds, img_root, conf_thr, iou_thr, max_images=None, device_id=0):
    mp = map_yolo_cls_to_classidx(yolo_model)
    per_class_all = [[] for _ in range(K)]

    n = len(ds) if max_images is None else min(len(ds), max_images)
    for i in tqdm(range(n), desc="Eval YOLO (TEST)"):
        img, tgt = ds[i]
        img_id = int(tgt["image_id"].item())
        fn = ds.coco.loadImgs(img_id)[0]["file_name"]
        img_path = os.path.join(img_root, fn)

        gt_boxes = tgt["boxes"].numpy()
        gt_labels_raw = tgt["labels"].numpy().tolist()

        mask = []
        gt_cls = []
        for lab in gt_labels_raw:
            ci = frcnn_label_to_classidx(lab, ds)
            mask.append(ci is not None)
            if ci is not None:
                gt_cls.append(ci)
        mask = np.array(mask, dtype=bool)
        gt_boxes = gt_boxes[mask]

        r = yolo_model.predict(img_path, conf=conf_thr, verbose=False, device=device_id)[0]
        if r.boxes is None or len(r.boxes) == 0:
            pb = np.zeros((0,4), dtype=np.float32)
            ps = []
            pl = []
        else:
            pb = r.boxes.xyxy.cpu().numpy()
            ps = r.boxes.conf.cpu().numpy().tolist()
            ycls = r.boxes.cls.cpu().numpy().astype(int).tolist()
            pl = [mp.get(c, None) for c in ycls]

        per_class_vals = compute_image_ious_per_class(gt_boxes, gt_cls, pb, pl, ps, conf_thr, iou_thr)
        for c in range(K):
            per_class_all[c].extend(per_class_vals[c])

    cls_means = [float(np.mean(v)) if len(v) else np.nan for v in per_class_all]
    valid = [m for m in cls_means if not np.isnan(m)]
    miou_macro = float(np.mean(valid)) if len(valid) else 0.0

    all_vals = [x for v in per_class_all for x in v]
    mean_all = float(np.mean(all_vals)) if len(all_vals) else 0.0
    return mean_all, miou_macro

@torch.no_grad()
def eval_miou_frcnn(frcnn_model, detector_num_classes, ds, img_root, device, conf_thr, iou_thr, max_images=None):
    per_class_all = [[] for _ in range(K)]

    n = len(ds) if max_images is None else min(len(ds), max_images)
    for i in tqdm(range(n), desc="Eval FRCNN (TEST)"):
        img, tgt = ds[i]
        gt_boxes = tgt["boxes"].numpy()
        gt_labels_raw = tgt["labels"].numpy().tolist()

        mask = []
        gt_cls = []
        for lab in gt_labels_raw:
            ci = frcnn_label_to_classidx(lab, ds)
            mask.append(ci is not None)
            if ci is not None:
                gt_cls.append(ci)
        mask = np.array(mask, dtype=bool)
        gt_boxes = gt_boxes[mask]

        out = frcnn_model([img.to(device)])[0]
        pb = out["boxes"].detach().cpu().numpy()
        ps = out["scores"].detach().cpu().numpy().tolist()
        pl_raw = out["labels"].detach().cpu().numpy().tolist()
        pl = [frcnn_label_to_classidx(lab, ds) for lab in pl_raw]

        per_class_vals = compute_image_ious_per_class(gt_boxes, gt_cls, pb, pl, ps, conf_thr, iou_thr)
        for c in range(K):
            per_class_all[c].extend(per_class_vals[c])

    cls_means = [float(np.mean(v)) if len(v) else np.nan for v in per_class_all]
    valid = [m for m in cls_means if not np.isnan(m)]
    miou_macro = float(np.mean(valid)) if len(valid) else 0.0

    all_vals = [x for v in per_class_all for x in v]
    mean_all = float(np.mean(all_vals)) if len(all_vals) else 0.0
    return mean_all, miou_macro

# run the comparison
rows = []

y8 = YOLO(YOLO8_BEST)
m_all, m_macro = eval_miou_yolo(y8, EVAL_DS, IMG_ROOT, CONF_THR, IOU_THR, MAX_IMAGES, device_id=0)
rows.append({"model":"yolov8s", "ckpt":YOLO8_BEST, "MeanIoU_allGT":m_all, "mIoU_macro":m_macro})
del y8; torch.cuda.empty_cache(); gc.collect()

y11 = YOLO(YOLO11_BEST)
m_all, m_macro = eval_miou_yolo(y11, EVAL_DS, IMG_ROOT, CONF_THR, IOU_THR, MAX_IMAGES, device_id=0)
rows.append({"model":"yolov11s", "ckpt":YOLO11_BEST, "MeanIoU_allGT":m_all, "mIoU_macro":m_macro})
del y11; torch.cuda.empty_cache(); gc.collect()

frcnn, frcnn_nc = load_frcnn_ckpt(FRCNN_BEST, device)
m_all, m_macro = eval_miou_frcnn(frcnn, frcnn_nc, EVAL_DS, IMG_ROOT, device, CONF_THR, IOU_THR, MAX_IMAGES)
rows.append({"model":"frcnn", "ckpt":FRCNN_BEST, "MeanIoU_allGT":m_all, "mIoU_macro":m_macro})
del frcnn; torch.cuda.empty_cache(); gc.collect()

df = pd.DataFrame(rows).sort_values("MeanIoU_allGT", ascending=False)
print(df)

out_csv = os.path.join(EXP_LOG_DIR, f"compare_3models_TEST_mIoU_iou{IOU_THR}_conf{CONF_THR}.csv")
df.to_csv(out_csv, index=False)
print("saved:", out_csv)

In [ ]:
import time, os, gc, torch
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from torchvision.ops import box_iou
from ultralytics import YOLO
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def cuda_sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

@torch.no_grad()
def timed_yolo_predict(yolo_model, img_np, conf_thr=0.25, device_id=0):
    cuda_sync()
    t0 = time.perf_counter()
    r = yolo_model.predict(img_np, conf=conf_thr, verbose=False, device=device_id)[0]
    cuda_sync()
    t1 = time.perf_counter()
    return r, (t1 - t0)

@torch.no_grad()
def timed_frcnn_predict(frcnn_model, img_t, device):
    img_t = img_t.to(device, non_blocking=True)
    cuda_sync()
    t0 = time.perf_counter()
    out = frcnn_model([img_t])[0]
    cuda_sync()
    t1 = time.perf_counter()
    return out, (t1 - t0)

def compute_per_class_ious(gt_boxes, gt_cls, pb, pl, ps, conf_thr, iou_thr, K):

    keep = [i for i,s in enumerate(ps) if (s >= conf_thr and pl[i] is not None)]
    pb = pb[keep] if len(keep) else np.zeros((0,4), dtype=np.float32)
    pl = [pl[i] for i in keep]
    ps = [ps[i] for i in keep]

    per_class_vals = [[] for _ in range(K)]
    for c in range(K):
        gt_idx = np.where(np.array(gt_cls) == c)[0]
        if gt_idx.size == 0:
            continue

        vals = [0.0] * int(gt_idx.size)
        pr_idx = [j for j,pc in enumerate(pl) if pc == c]
        if len(pr_idx) > 0:
            gtb = torch.tensor(gt_boxes[gt_idx], dtype=torch.float32)
            prb = torch.tensor(pb[pr_idx], dtype=torch.float32)
            iou_mat = box_iou(prb, gtb)

            used_p=set(); used_g=set()
            flat = iou_mat.flatten()
            order = torch.argsort(flat, descending=True)
            P,G = iou_mat.shape

            for idx_flat in order.tolist():
                p = idx_flat // G
                g = idx_flat %  G
                if p in used_p or g in used_g:
                    continue
                v = float(iou_mat[p,g].item())
                if v < iou_thr:
                    break
                used_p.add(p); used_g.add(g)
                vals[g] = v

        per_class_vals[c].extend(vals)

    return per_class_vals

K = len(CLASS_NAMES)
name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}

def map_yolo_cls_to_classidx(yolo_model):
    mp = {}
    for yid, nm in yolo_model.names.items():
        mp[int(yid)] = name_to_idx.get(nm, None)
    return mp

def frcnn_label_to_classidx(label_contig, ds):
    lab = int(label_contig)
    if hasattr(ds, "contig_to_catid") and lab in ds.contig_to_catid:
        catid = ds.contig_to_catid[lab]
        nm = ds.coco.cats.get(catid, {}).get("name", None)
        if nm in name_to_idx:
            return name_to_idx[nm]
    x = lab - 1
    return x if 0 <= x < K else None

def load_frcnn_ckpt(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt.get("model_state_dict", ckpt.get("model_state", None))
    if state is None:
        raise KeyError("FRCNN ckpt missing model_state_dict/model_state")
    detector_num_classes = int(ckpt.get("detector_num_classes", K + 1))
    MIN_SIZE = int(ckpt.get("MIN_SIZE", 600))
    MAX_SIZE = int(ckpt.get("MAX_SIZE", 1000))

    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone=None, min_size=MIN_SIZE, max_size=MAX_SIZE
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, detector_num_classes)
    model.load_state_dict(state)
    model.to(device).eval()
    return model

def eval_test_yolo_miou_and_time(best_pt, ds, img_root, conf_thr=0.25, iou_thr=0.5, max_images=None, warmup=10):
    model = YOLO(best_pt)
    mp = map_yolo_cls_to_classidx(model)

    per_class_all = [[] for _ in range(K)]
    total_inf_time = 0.0
    n = len(ds) if max_images is None else min(len(ds), max_images)

    for i in range(min(warmup, n)):
        img, _ = ds[i]
        img_np = (img.permute(1,2,0).numpy() * 255).astype(np.uint8)
        _ = model.predict(img_np, conf=conf_thr, verbose=False, device=0)

    for i in tqdm(range(n), desc=f"YOLO test ({os.path.basename(best_pt)})"):
        img, tgt = ds[i]
        img_id = int(tgt["image_id"].item())
        fn = ds.coco.loadImgs(img_id)[0]["file_name"]
        img_path = os.path.join(img_root, fn)

        gt_boxes = tgt["boxes"].numpy()
        gt_labels_raw = tgt["labels"].numpy().tolist()

        mask = []
        gt_cls = []
        for lab in gt_labels_raw:
            ci = frcnn_label_to_classidx(lab, ds)
            mask.append(ci is not None)
            if ci is not None:
                gt_cls.append(ci)
        mask = np.array(mask, dtype=bool)
        gt_boxes = gt_boxes[mask]

        img_np = (img.permute(1,2,0).numpy() * 255).astype(np.uint8)

        r, dt = timed_yolo_predict(model, img_np, conf_thr=conf_thr, device_id=0)
        total_inf_time += dt

        if r.boxes is None or len(r.boxes) == 0:
            pb = np.zeros((0,4), dtype=np.float32)
            ps = []
            pl = []
        else:
            pb = r.boxes.xyxy.cpu().numpy()
            ps = r.boxes.conf.cpu().numpy().tolist()
            ycls = r.boxes.cls.cpu().numpy().astype(int).tolist()
            pl = [mp.get(c, None) for c in ycls]

        per_class_vals = compute_per_class_ious(gt_boxes, gt_cls, pb, pl, ps, conf_thr, iou_thr, K)
        for c in range(K):
            per_class_all[c].extend(per_class_vals[c])

    cls_means = [float(np.mean(v)) if len(v) else np.nan for v in per_class_all]
    valid = [m for m in cls_means if not np.isnan(m)]
    miou_macro = float(np.mean(valid)) if len(valid) else 0.0
    all_vals = [x for v in per_class_all for x in v]
    mean_all = float(np.mean(all_vals)) if len(all_vals) else 0.0

    return mean_all, miou_macro, total_inf_time, n

def eval_test_frcnn_miou_and_time(best_ckpt, ds, img_root, conf_thr=0.25, iou_thr=0.5, max_images=None, warmup=10):
    model = load_frcnn_ckpt(best_ckpt, device)

    per_class_all = [[] for _ in range(K)]
    total_inf_time = 0.0
    n = len(ds) if max_images is None else min(len(ds), max_images)

    for i in range(min(warmup, n)):
        img, _ = ds[i]
        _ = model([img.to(device)])[0]

    for i in tqdm(range(n), desc=f"FRCNN test ({os.path.basename(best_ckpt)})"):
        img, tgt = ds[i]
        gt_boxes = tgt["boxes"].numpy()
        gt_labels_raw = tgt["labels"].numpy().tolist()

        mask = []
        gt_cls = []
        for lab in gt_labels_raw:
            ci = frcnn_label_to_classidx(lab, ds)
            mask.append(ci is not None)
            if ci is not None:
                gt_cls.append(ci)
        mask = np.array(mask, dtype=bool)
        gt_boxes = gt_boxes[mask]

        out, dt = timed_frcnn_predict(model, img, device)
        total_inf_time += dt

        pb = out["boxes"].detach().cpu().numpy()
        ps = out["scores"].detach().cpu().numpy().tolist()
        pl_raw = out["labels"].detach().cpu().numpy().tolist()
        pl = [frcnn_label_to_classidx(lab, ds) for lab in pl_raw]

        per_class_vals = compute_per_class_ious(gt_boxes, gt_cls, pb, pl, ps, conf_thr, iou_thr, K)
        for c in range(K):
            per_class_all[c].extend(per_class_vals[c])

    cls_means = [float(np.mean(v)) if len(v) else np.nan for v in per_class_all]
    valid = [m for m in cls_means if not np.isnan(m)]
    miou_macro = float(np.mean(valid)) if len(valid) else 0.0
    all_vals = [x for v in per_class_all for x in v]
    mean_all = float(np.mean(all_vals)) if len(all_vals) else 0.0

    del model
    torch.cuda.empty_cache()
    gc.collect()

    return mean_all, miou_macro, total_inf_time, n

YOLO8_BEST  = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments/runs/yolov8s_11000/weights/best.pt"
YOLO11_BEST = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments/runs/yolo11s_11000/weights/best.pt"
FRCNN_BEST  = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments_frcnn/best_frcnn11000.pt"

CONF_THR = 0.75
IOU_THR  = 0.75
MAX_IMAGES = None

rows = []

m_all, m_macro, tsec, n = eval_test_yolo_miou_and_time(YOLO8_BEST, test_ds, TEST_IMG_DIR, CONF_THR, IOU_THR, MAX_IMAGES)
rows.append({"model":"yolov8s", "MeanIoU_allGT":m_all, "mIoU_macro":m_macro,
             "test_imgs":n, "infer_total_sec":tsec, "avg_ms_per_img":(tsec/n)*1000, "fps":n/max(tsec,1e-9)})

m_all, m_macro, tsec, n = eval_test_yolo_miou_and_time(YOLO11_BEST, test_ds, TEST_IMG_DIR, CONF_THR, IOU_THR, MAX_IMAGES)
rows.append({"model":"yolov11s", "MeanIoU_allGT":m_all, "mIoU_macro":m_macro,
             "test_imgs":n, "infer_total_sec":tsec, "avg_ms_per_img":(tsec/n)*1000, "fps":n/max(tsec,1e-9)})

m_all, m_macro, tsec, n = eval_test_frcnn_miou_and_time(FRCNN_BEST, test_ds, TEST_IMG_DIR, CONF_THR, IOU_THR, MAX_IMAGES)
rows.append({"model":"frcnn", "MeanIoU_allGT":m_all, "mIoU_macro":m_macro,
             "test_imgs":n, "infer_total_sec":tsec, "avg_ms_per_img":(tsec/n)*1000, "fps":n/max(tsec,1e-9)})

df = pd.DataFrame(rows).sort_values("infer_total_sec")
print(df)

out_csv = os.path.join(EXP_LOG_DIR, f"compare_3models_TEST_mIoU_time_iou{IOU_THR}_conf{CONF_THR}.csv")
df.to_csv(out_csv, index=False)
print("saved:", out_csv)

In [ ]:
# compare the three models on the test set
import os, time, gc
import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from torchvision.ops import box_iou
from pycocotools.cocoeval import COCOeval
from ultralytics import YOLO
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# paths
YOLO8_BEST  = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments/runs/yolov8s_11000/weights/best.pt"
YOLO11_BEST = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments/runs/yolo11s_11000/weights/best.pt"
FRCNN_BEST  = "/content/drive/Shareddrives/Workspace/NKla/AI_TECH_PROJECT/Experiments_frcnn/best_frcnn11000.pt"

# thresholds
CONF_THR   = 0.5
IOU_MATCH  = 0.5
MAX_IMAGES = None

# optional limit for Faster R-CNN loss calculation
LOSS_MAX_BATCHES = None  # set None to compute on all test batches

CLASS_NAMES = CLASS_NAMES
K = len(CLASS_NAMES)

def cuda_sync():
    if device.type == "cuda":
        torch.cuda.synchronize()

# small COCO ground-truth fix
def fix_coco_gt(coco):
    changed = False
    for a in coco.dataset.get("annotations", []):
        if "iscrowd" not in a:
            a["iscrowd"] = 0
            changed = True
        if "area" not in a and "bbox" in a:
            a["area"] = float(a["bbox"][2] * a["bbox"][3])
            changed = True
    if changed:
        coco.createIndex()
    return coco

coco_gt = fix_coco_gt(test_ds.coco)

# map class names back to COCO category ids
cat_name_to_id = {c["name"]: c["id"] for c in coco_gt.dataset.get("categories", [])}
name_to_idx = {n: i for i, n in enumerate(CLASS_NAMES)}

# helper: plot confusion matrix
def plot_cm(cm, labels, title, save_path):
    plt.figure(figsize=(10,8))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.yticks(range(len(labels)), labels)
    plt.colorbar()
    plt.tight_layout()
    plt.savefig(save_path, dpi=200)
    plt.show()
    print("saved:", save_path)

# matching logic for precision, recall, and confusion matrix
def pr_and_cm_from_detections(gt_boxes, gt_cls_idx, pb, pl_idx, ps,
                             conf_thr=0.5, iou_thr=0.5, K=11):
    """
    gt_cls_idx, pl_idx are 0..K-1 (CLASS_NAMES index). pl_idx can include None.
    returns: TP, FP, FN, cm (K+1 x K+1 with BG at last index)
    """
    BG = K
    cm = np.zeros((K+1, K+1), dtype=np.int64)

    # filter preds
    keep = [i for i,s in enumerate(ps) if (s >= conf_thr and pl_idx[i] is not None)]
    pb = pb[keep] if len(keep) else np.zeros((0,4), dtype=np.float32)
    pl = [pl_idx[i] for i in keep]
    ps = [ps[i] for i in keep]

    # sort by score desc
    if len(ps) > 0:
        order = np.argsort(ps)[::-1]
        pb = pb[order]
        pl = [pl[i] for i in order]
        ps = [ps[i] for i in order]

    # per image matching (class-aware greedy)
    TP = FP = 0
    matched_gt = np.zeros((len(gt_boxes),), dtype=bool)

    if len(pb) > 0 and len(gt_boxes) > 0:
        ious = box_iou(torch.tensor(pb, dtype=torch.float32),
                       torch.tensor(gt_boxes, dtype=torch.float32)).numpy()
    else:
        ious = None

    for p in range(len(pb)):
        pred_c = pl[p]

        if len(gt_boxes) == 0:
            FP += 1
            cm[BG, pred_c] += 1
            continue

        # only consider GT with same class and not matched yet
        valid_g = [g for g in range(len(gt_boxes))
                   if (not matched_gt[g]) and (gt_cls_idx[g] == pred_c)]
        if len(valid_g) == 0:
            FP += 1
            cm[BG, pred_c] += 1
            continue

        best_g = max(valid_g, key=lambda g: ious[p, g])
        best_iou = ious[p, best_g]

        if best_iou >= iou_thr:
            TP += 1
            matched_gt[best_g] = True
            cm[gt_cls_idx[best_g], pred_c] += 1
        else:
            FP += 1
            cm[BG, pred_c] += 1

    # FN: GT not matched
    FN = int((~matched_gt).sum())
    for g in range(len(gt_boxes)):
        if not matched_gt[g]:
            cm[gt_cls_idx[g], BG] += 1

    return TP, FP, FN, cm

# 5) COCOeval for mAP50 / mAP50-95
def coco_eval_from_results(coco_gt, coco_results):
    if len(coco_results) == 0:
        return {"mAP50_95": 0.0, "mAP50": 0.0}

    coco_dt = coco_gt.loadRes(coco_results)
    coco_eval = COCOeval(coco_gt, coco_dt, iouType="bbox")
    coco_eval.evaluate()
    coco_eval.accumulate()
    coco_eval.summarize()
    s = coco_eval.stats
    return {"mAP50_95": float(s[0]), "mAP50": float(s[1])}

# 6) YOLO: run test -> coco_results + PR + CM + time + read losses from results.csv
def yolo_results_csv_path(best_pt):

    wdir = os.path.dirname(best_pt)
    rdir = os.path.dirname(wdir)
    p = os.path.join(rdir, "results.csv")
    return p if os.path.exists(p) else None

def read_yolo_best_losses(results_csv):

    if results_csv is None:
        return (np.nan, np.nan, np.nan)
    df = pd.read_csv(results_csv)
    map_col = None
    for c in ["metrics/mAP50-95(B)", "metrics/mAP50-95", "metrics/mAP50-95(box)"]:
        if c in df.columns:
            map_col = c
            break
    if map_col is None:
        return (np.nan, np.nan, np.nan)

    best_row = df.loc[df[map_col].idxmax()]

    def pick(colnames):
        for c in colnames:
            if c in df.columns:
                return float(best_row[c])
        return np.nan

    box_loss = pick(["train/box_loss", "box_loss"])
    cls_loss = pick(["train/cls_loss", "cls_loss"])
    dfl_loss = pick(["train/dfl_loss", "dfl_loss"])
    return box_loss, cls_loss, dfl_loss

@torch.no_grad()
def eval_yolo_on_test(best_pt, ds, img_root, coco_gt, conf_thr=0.5, iou_match=0.5, max_images=None):
    model = YOLO(best_pt)

    yolo_id_to_name = model.names
    yolo_id_to_catid = {int(i): cat_name_to_id.get(nm, None) for i, nm in yolo_id_to_name.items()}
    yolo_id_to_clsidx = {int(i): name_to_idx.get(nm, None) for i, nm in yolo_id_to_name.items()}

    coco_results = []
    TP=FP=FN=0
    cm_sum = np.zeros((K+1, K+1), dtype=np.int64)

    total_inf = 0.0
    n = len(ds) if max_images is None else min(len(ds), max_images)

    for i in range(min(10, n)):
        img, _ = ds[i]
        img_np = (img.permute(1,2,0).numpy() * 255).astype(np.uint8)
        _ = model.predict(img_np, conf=conf_thr, verbose=False, device=0)

    for i in tqdm(range(n), desc=f"YOLO test ({os.path.basename(best_pt)})"):
        img, tgt = ds[i]
        img_id = int(tgt["image_id"].item())
        fn = ds.coco.loadImgs(img_id)[0]["file_name"]
        img_path = os.path.join(img_root, fn)

        gt_boxes = tgt["boxes"].numpy()
        gt_labels_raw = tgt["labels"].numpy().tolist()

        gt_cls = []
        gt_keep = []
        for lab in gt_labels_raw:
            ci = lab - 1
            if 0 <= ci < K:
                gt_cls.append(ci)
                gt_keep.append(True)
            else:
                gt_keep.append(False)
        gt_boxes = gt_boxes[np.array(gt_keep, dtype=bool)]

        img_np = (img.permute(1,2,0).numpy() * 255).astype(np.uint8)
        cuda_sync()
        t0 = time.perf_counter()
        r = model.predict(img_np, conf=conf_thr, verbose=False, device=0)[0]
        cuda_sync()
        total_inf += (time.perf_counter() - t0)

        if r.boxes is None or len(r.boxes) == 0:
            pb = np.zeros((0,4), dtype=np.float32)
            ps = []
            pl_idx = []
            ycls = []
        else:
            pb = r.boxes.xyxy.cpu().numpy()
            ps = r.boxes.conf.cpu().numpy().tolist()
            ycls = r.boxes.cls.cpu().numpy().astype(int).tolist()
            pl_idx = [yolo_id_to_clsidx.get(c, None) for c in ycls]

            # for COCOeval results
            for box, score, yc in zip(pb, ps, ycls):
                catid = yolo_id_to_catid.get(int(yc), None)
                if catid is None or score < conf_thr:
                    continue
                x1,y1,x2,y2 = box.tolist()
                w,h = x2-x1, y2-y1
                if w<=0 or h<=0:
                    continue
                coco_results.append({
                    "image_id": img_id,
                    "category_id": int(catid),
                    "bbox": [x1,y1,w,h],
                    "score": float(score)
                })

        tp, fp, fn, cm = pr_and_cm_from_detections(gt_boxes, gt_cls, pb, pl_idx, ps, conf_thr, iou_match, K)
        TP += tp; FP += fp; FN += fn
        cm_sum += cm

    precision = TP / max(TP+FP, 1)
    recall    = TP / max(TP+FN, 1)

    # COCO mAP
    maps = coco_eval_from_results(coco_gt, coco_results)

    # losses from results.csv (if exists)
    rcsv = yolo_results_csv_path(best_pt)
    box_loss, cls_loss, dfl_loss = read_yolo_best_losses(rcsv)

    return {
        "precision": float(precision),
        "recall": float(recall),
        "mAP50": maps["mAP50"],
        "mAP50_95": maps["mAP50_95"],
        "box_loss": box_loss,
        "cls_loss": cls_loss,
        "dfl_loss": dfl_loss,
        "infer_total_sec": float(total_inf),
        "imgs": int(n),
        "cm": cm_sum,
        "results_csv": rcsv
    }

def load_frcnn_ckpt(ckpt_path, device):
    ckpt = torch.load(ckpt_path, map_location=device)
    state = ckpt.get("model_state_dict", ckpt.get("model_state", None))
    if state is None:
        raise KeyError("FRCNN ckpt missing model_state_dict/model_state")
    detector_num_classes = int(ckpt.get("detector_num_classes", K+1))
    MIN_SIZE = int(ckpt.get("MIN_SIZE", 600))
    MAX_SIZE = int(ckpt.get("MAX_SIZE", 1000))

    model = torchvision.models.detection.fasterrcnn_resnet50_fpn(
        weights=None, weights_backbone=None, min_size=MIN_SIZE, max_size=MAX_SIZE
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, detector_num_classes)
    model.load_state_dict(state)
    model.to(device).eval()
    return model, detector_num_classes

@torch.no_grad()
def eval_frcnn_on_test(ckpt_path, ds, loader, coco_gt, conf_thr=0.5, iou_match=0.5, max_images=None):
    model, detector_num_classes = load_frcnn_ckpt(ckpt_path, device)

    coco_results = []
    TP=FP=FN=0
    cm_sum = np.zeros((K+1, K+1), dtype=np.int64)
    total_inf = 0.0

    n = len(ds) if max_images is None else min(len(ds), max_images)

    for i in range(min(10, n)):
        img, _ = ds[i]
        _ = model([img.to(device)])[0]

    for i in tqdm(range(n), desc="FRCNN test"):
        img, tgt = ds[i]
        img_id = int(tgt["image_id"].item())

        gt_boxes = tgt["boxes"].numpy()
        gt_labels_raw = tgt["labels"].numpy().tolist()
        gt_cls = []
        gt_keep=[]
        for lab in gt_labels_raw:
            ci = lab - 1
            if 0 <= ci < K:
                gt_cls.append(ci); gt_keep.append(True)
            else:
                gt_keep.append(False)
        gt_boxes = gt_boxes[np.array(gt_keep, dtype=bool)]

        cuda_sync()
        t0 = time.perf_counter()
        out = model([img.to(device)])[0]
        cuda_sync()
        total_inf += (time.perf_counter() - t0)

        pb = out["boxes"].detach().cpu().numpy()
        ps = out["scores"].detach().cpu().numpy().tolist()
        pl_raw = out["labels"].detach().cpu().numpy().tolist()
        pl_idx = []
        for lab in pl_raw:
            ci = int(lab) - 1
            pl_idx.append(ci if 0 <= ci < K else None)

        for box, score, lab in zip(pb, ps, pl_raw):
            if score < conf_thr:
                continue
            catid = ds.contig_to_catid.get(int(lab), None) if hasattr(ds, "contig_to_catid") else None
            if catid is None:
                continue
            x1,y1,x2,y2 = box.tolist()
            w,h = x2-x1, y2-y1
            if w<=0 or h<=0:
                continue
            coco_results.append({
                "image_id": img_id,
                "category_id": int(catid),
                "bbox": [x1,y1,w,h],
                "score": float(score)
            })

        tp, fp, fn, cm = pr_and_cm_from_detections(gt_boxes, gt_cls, pb, pl_idx, ps, conf_thr, iou_match, K)
        TP += tp; FP += fp; FN += fn
        cm_sum += cm

    precision = TP / max(TP+FP, 1)
    recall    = TP / max(TP+FN, 1)
    maps = coco_eval_from_results(coco_gt, coco_results)

    model.train()
    loss_box = loss_cls = np.nan
    lsum_box = 0.0
    lsum_cls = 0.0
    cnt = 0
    with torch.no_grad():
        for b, (images, targets) in enumerate(tqdm(loader, desc="FRCNN loss (test)")):
            if LOSS_MAX_BATCHES is not None and b >= LOSS_MAX_BATCHES:
                break
            images = [im.to(device) for im in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            loss_dict = model(images, targets)

            lcls = float(loss_dict.get("loss_classifier", 0.0).item())
            lbox = float(loss_dict.get("loss_box_reg", 0.0).item())
            lsum_cls += lcls
            lsum_box += lbox
            cnt += 1
            del loss_dict, images, targets
    if cnt > 0:
        loss_cls = lsum_cls / cnt
        loss_box = lsum_box / cnt

    model.eval()
    del model
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "precision": float(precision),
        "recall": float(recall),
        "mAP50": maps["mAP50"],
        "mAP50_95": maps["mAP50_95"],
        "box_loss": float(loss_box),
        "cls_loss": float(loss_cls),
        "dfl_loss": np.nan,
        "infer_total_sec": float(total_inf),
        "imgs": int(n),
        "cm": cm_sum
    }

rows = []

res8 = eval_yolo_on_test(YOLO8_BEST, test_ds, TEST_IMG_DIR, coco_gt, CONF_THR, IOU_MATCH, MAX_IMAGES)
rows.append({"model":"yolov8s", **{k:v for k,v in res8.items() if k!="cm"}})
cm8 = res8["cm"]

res11 = eval_yolo_on_test(YOLO11_BEST, test_ds, TEST_IMG_DIR, coco_gt, CONF_THR, IOU_MATCH, MAX_IMAGES)
rows.append({"model":"yolov11s", **{k:v for k,v in res11.items() if k!="cm"}})
cm11 = res11["cm"]

resf = eval_frcnn_on_test(FRCNN_BEST, test_ds, test_loader, coco_gt, CONF_THR, IOU_MATCH, MAX_IMAGES)
rows.append({"model":"frcnn", **{k:v for k,v in resf.items() if k!="cm"}})
cmf = resf["cm"]

df = pd.DataFrame(rows)

df["avg_ms_per_img"] = (df["infer_total_sec"] / df["imgs"]) * 1000.0
df["fps"] = df["imgs"] / df["infer_total_sec"].clip(lower=1e-9)

display(df)
csv_path = os.path.join(EXP_LOG_DIR, f"compare_3models_TEST_metrics_conf{CONF_THR}_iou{IOU_MATCH}.csv")
df.to_csv(csv_path, index=False)
print("saved:", csv_path)

labels = list(CLASS_NAMES) + ["__background__"]
plot_cm(cm8,  labels, f"YOLOv8s Confusion (conf>={CONF_THR}, iou>={IOU_MATCH})",
        os.path.join(EXP_LOG_DIR, f"cm_yolov8s_conf{CONF_THR}_iou{IOU_MATCH}.png"))
plot_cm(cm11, labels, f"YOLOv11s Confusion (conf>={CONF_THR}, iou>={IOU_MATCH})",
        os.path.join(EXP_LOG_DIR, f"cm_yolov11s_conf{CONF_THR}_iou{IOU_MATCH}.png"))
plot_cm(cmf,  labels, f"FRCNN Confusion (conf>={CONF_THR}, iou>={IOU_MATCH})",
        os.path.join(EXP_LOG_DIR, f"cm_frcnn_conf{CONF_THR}_iou{IOU_MATCH}.png"))

In [ ]:
# save the confusion matrices
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

labels = list(CLASS_NAMES) + ["__background__"]

def save_cm_csv(cm, name):
    path = os.path.join(EXP_LOG_DIR, f"cm_{name}_raw.csv")
    pd.DataFrame(cm, index=labels, columns=labels).to_csv(path)
    print("saved:", path)
    return path

def plot_cm_numbers(cm, title, save_path, normalize=False):
    cm = cm.astype(np.float64)

    if normalize:
        row_sum = cm.sum(axis=1, keepdims=True)
        cm = np.divide(cm, row_sum, out=np.zeros_like(cm), where=row_sum != 0)

    plt.figure(figsize=(12, 10))
    plt.imshow(cm, interpolation="nearest")
    plt.title(title)
    plt.xticks(range(len(labels)), labels, rotation=90)
    plt.yticks(range(len(labels)), labels)
    plt.colorbar()
    plt.tight_layout()

    # write the value in each cell
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            val = cm[i, j]
            txt = f"{val:.2f}" if normalize else f"{int(val)}"
            # pick text color based on the background
            color = "white" if val > (cm.max() * 0.5) else "black"
            plt.text(j, i, txt, ha="center", va="center", color=color, fontsize=7)

    plt.savefig(save_path, dpi=220, bbox_inches="tight")
    plt.show()
    print("saved:", save_path)

def top_confusions(cm, name, topk=15, ignore_bg=True):
    cm = cm.copy()
    K = len(CLASS_NAMES)
    BG = K

    # skip the diagonal
    np.fill_diagonal(cm, 0)

    if ignore_bg:
        # skip the background row and column
        cm[BG, :] = 0
        cm[:, BG] = 0

    pairs = []
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            if cm[i, j] > 0:
                pairs.append((int(cm[i, j]), labels[i], labels[j]))

    pairs.sort(reverse=True, key=lambda x: x[0])
    df = pd.DataFrame(pairs[:topk], columns=["count", "GT", "Pred"])
    print(f"\nTop confusions ({name})")
    display(df)
    return df

# these matrices should already exist from the previous comparison cell
# cm8, cm11, cmf

# YOLOv8s
save_cm_csv(cm8, "yolov8s")
plot_cm_numbers(cm8, "YOLOv8s Confusion Matrix (raw counts)",
                os.path.join(EXP_LOG_DIR, "cm_yolov8s_numbers_raw.png"), normalize=False)
plot_cm_numbers(cm8, "YOLOv8s Confusion Matrix (row-normalized)",
                os.path.join(EXP_LOG_DIR, "cm_yolov8s_numbers_norm.png"), normalize=True)
top_confusions(cm8, "yolov8s", topk=15, ignore_bg=True)

# YOLOv11s
save_cm_csv(cm11, "yolov11s")
plot_cm_numbers(cm11, "YOLOv11s Confusion Matrix (raw counts)",
                os.path.join(EXP_LOG_DIR, "cm_yolov11s_numbers_raw.png"), normalize=False)
plot_cm_numbers(cm11, "YOLOv11s Confusion Matrix (row-normalized)",
                os.path.join(EXP_LOG_DIR, "cm_yolov11s_numbers_norm.png"), normalize=True)
top_confusions(cm11, "yolov11s", topk=15, ignore_bg=True)

# Faster R-CNN
save_cm_csv(cmf, "frcnn")
plot_cm_numbers(cmf, "FRCNN Confusion Matrix (raw counts)",
                os.path.join(EXP_LOG_DIR, "cm_frcnn_numbers_raw.png"), normalize=False)
plot_cm_numbers(cmf, "FRCNN Confusion Matrix (row-normalized)",
                os.path.join(EXP_LOG_DIR, "cm_frcnn_numbers_norm.png"), normalize=True)
top_confusions(cmf, "frcnn", topk=15, ignore_bg=True)